In [0]:
from datetime import date, datetime 
from pyspark.sql.functions import col, to_date, current_timestamp, lit

#User pass date 
try:
    arrival_date=dbutils.widgets.get("arrival_date")
except Exception:
    arrival_date=date.today().strftime("%Y-%m-%d")

#User pass catalog
try:
	catalog=dbutils.widgets.get("catalog")
except Exception:
	catalog="travel_bookings"

#User pass schema
try:
	schema=dbutils.widgets.get("schema")
except Exception:
	schema="default"

In [0]:
#Read bronze layaer booking table 
booking_table=f"{catalog}.bronze.booking_inc"
#Create schema
spark.sql(f"create schema if not exists {catalog}.ops")

from pydeequ.checks import Check, CheckLevel
from pydeequ.verification import VerificationSuite, VerificationResult

spark.sql(f"""create table if not exists {catalog}.ops.dq_results 
(business_date date, dataset string, check_name string, status string, constraint string,
message string, recorded_at timestamp) using delta
""")

df_booking=spark.read.table(booking_table)

src=df_booking.filter(col("business_date")==datetime.strptime(arrival_date, "%Y-%m-%d").date())

#Create checks using pydeequ library
check=Check(spark, CheckLevel.Error, "Data Quality checks")\
.hasSize(lambda x: x > 0)\
.isComplete("customer_id")\
.isComplete("amount")\
.isNonNegative("amount")\
.isNonNegative("quantity")\
.isNonNegative("discount")

#Run check against dataframe
result=VerificationSuite(spark)\
.onData(src)\
.addCheck(check)\
.run()

df=VerificationResult.checkResultsAsDataFrame(spark, result)
out=df.withColumn("business_date", to_date(lit("arrival_date"))).withColumn("dataset", lit("booking_inc"))\
.withColumn("recorded_at", current_timestamp())

out.select("business_date", "dataset", "check", "check_status", "constraint", "constraint_status",
"constraint_message", "recorded_at").write.mode('append').option('mergeSchema', 'true').saveAsTable(f"{catalog}.ops.dq_results")

if result.status!="SUCCESS":
	raise ValueError("DQ failed for bookings")
else:
	print("Booking DQ passed")